# MLflow + LLMs Lesson — BigCodeBench Tasks

We will use local LLMs (via **Ollama**) to solve a small set of coding tasks inspired by BigCodeBench, and track everything with MLflow on DagsHub:

1. **LLM parameters** — model name, temperature, max tokens
2. **Prompt evolution** — how the prompt affects output quality
3. **Pass@1 metric** — execute the generated code against test cases
4. **MLflow Tracing** — capture each LLM call with latency and token counts

## 1. Setup

In [ ]:
%pip install openai

In [ ]:
import re
import mlflow
import dagshub
import openai
import pandas as pd

## 2. Connect to DagsHub

In [ ]:
DAGSHUB_USER = "atdepo"
DAGSHUB_REPO = "SE4AI_2026_MLFlow_Lab"

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("llm-bigcodebench-4")

mlflow.autolog()
mlflow.openai.autolog(disable=True)  # openai autolog not yet compatible with DagsHub

## 3. BigCodeBench Tasks

We load the first 5 tasks from the BigCodeBench CSV. The columns we use are:
- `instruct_prompt` — the natural-language instruction for the LLM
- `entry_point` — the function name the LLM must produce
- `test` — the test code to evaluate correctness

In [ ]:
df_tasks = pd.read_csv("data/bigcodebench.csv").head(1)

TASKS = [
    {
        "task_id":     row["task_id"],
        "prompt":      row["instruct_prompt"],
        "entry_point": row["entry_point"],
        "test_code":   row["test"],
    }
    for _, row in df_tasks.iterrows()
]

print(f"{len(TASKS)} tasks loaded")
df_tasks[["task_id", "entry_point"]].head()

## 4. Helper Functions

### Code extraction
LLMs often wrap their answer in a markdown code block. We strip that to get runnable Python.

### Pass@1 evaluation
We `exec()` the generated code and then run the task's test assertions. If no exception is raised, the solution is correct — that's **Pass@1**: does the model solve the task on the first try?

In [ ]:
def extract_code(text):
    """Strip markdown fences and return only the Python code."""
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


def evaluate(generated_code, task):
    """Execute generated code + test assertions. Returns (passed: bool, error: str)."""
    namespace = {}
    try:
        exec(generated_code, namespace)
        if task["entry_point"] not in namespace:
            return False, f"Function '{task['entry_point']}' not found in output"
        exec(task["test_code"], namespace)
        return True, ""
    except Exception as e:
        return False, str(e)

## 5. LLM Client

The LLM is called via the **OpenAI-compatible REST API** that all major local servers expose on `localhost`. Change `BASE_URL` to match your server:
- **Ollama**: `http://localhost:11434/v1`
- **LM Studio**: `http://localhost:1234/v1`
- **llama.cpp**: `http://localhost:8080/v1`

**Note on autolog:** `mlflow.autolog()` is enabled in section 2. The openai flavour is explicitly disabled because it is not yet compatible with DagsHub — it tries to write traces to an endpoint DagsHub doesn't support yet, which breaks regular param logging. `mlflow.start_run()` in `run_sweep` is still needed: it is a *run context*, not a logging call — there is no framework hook to trigger an automatic run for custom LLM code. When `mlflow.openai.autolog()` gains DagsHub support, the line can be removed and it will capture inputs, outputs, and token counts automatically.

In [ ]:
BASE_URL = "http://localhost:1234/v1"
API_KEY  = "local"

client = openai.OpenAI(base_url=BASE_URL, api_key=API_KEY)


def call_llm(prompt, model, temperature, max_tokens, top_p, presence_penalty):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        presence_penalty=presence_penalty,
    )
    return response.choices[0].message.content

## 6. Experiments

We run one sweep per parameter, keeping the others fixed. This way we can isolate the effect of each parameter on pass@1.

**Fixed defaults:** `temperature=0.2`, `top_p=0.95`, `presence_penalty=0.0`, `max_tokens=512`

In [ ]:
MODEL    = "liquid/lfm2-1.2b"  # change to the model you have available locally

DEFAULTS = {"temperature": 0.2, "top_p": 0.95, "presence_penalty": 0.0, "max_tokens": 512}


def run_sweep(param_name, values):
    results = []
    for val in values:
        params = {**DEFAULTS, param_name: val}

        mlflow.set_tag("mlflow.runName", f"{param_name}={val}")
        mlflow.log_params({"model": MODEL, **params})

        passed = 0
        prompts, generated_codes, test_results = {}, {}, {}

        for task in TASKS:
            prompts[task["task_id"]] = task["prompt"]
            try:
                raw  = call_llm(task["prompt"], MODEL,
                                params["temperature"], params["max_tokens"],
                                params["top_p"], params["presence_penalty"])
                code = extract_code(raw)
                ok, err = evaluate(code, task)
            except openai.APIConnectionError as e:
                print(f"  [connection error] {e} — is the local server running?")
                ok, code, err = False, "", str(e)
            except Exception as e:
                print(f"  [error on {task['task_id']}] {e}")
                ok, code, err = False, "", str(e)

            passed += int(ok)
            generated_codes[task["task_id"]] = code
            test_results[task["task_id"]]    = {"passed": ok, "error": err}

        pass_at_1 = passed / len(TASKS)
        mlflow.log_metric("pass_at_1", pass_at_1)

        mlflow.log_text(
            "\n\n".join(f"# {tid}\n{p}" for tid, p in prompts.items()),
            "prompts.txt"
        )
        mlflow.log_text(
            "\n\n".join(f"# {tid}\n{code}" for tid, code in generated_codes.items()),
            "generated_code.py"
        )
        mlflow.log_text(
            "\n\n".join(
                f"# {tid}\nStatus: {'PASS' if r['passed'] else 'FAIL'}"
                + (f"\nError:  {r['error']}" if not r["passed"] else "")
                for tid, r in test_results.items()
            ),
            "test_results.txt"
        )

        mlflow.end_run()

        results.append({param_name: val, "pass_at_1": pass_at_1})
        print(f"  {param_name}={val:<6} | pass@1: {pass_at_1:.2f}")
    return results

### 6.1 Temperature Sweep

In [ ]:
print("--- Temperature sweep ---")
temp_results = run_sweep("temperature", [0.3, 0.5, 1.0])

### 6.2 Top-p Sweep

In [ ]:
print("--- Top-p sweep ---")
topp_results = run_sweep("top_p", [0.3, 0.5, 1.0])

### 6.3 Presence Penalty Sweep

In [ ]:
print("--- Presence penalty sweep ---")
penalty_results = run_sweep("presence_penalty", [-0.5, 0.0, 0.3, 0.6, 1.0])

## 8. Overall Comparison

In [ ]:
import matplotlib.pyplot as plt

sweeps = [
    ("temperature",      temp_results,    "temperature"),
    ("top_p",            topp_results,    "top_p"),
    ("presence_penalty", penalty_results, "presence_penalty"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (title, results, param) in zip(axes, sweeps):
    df = pd.DataFrame(results)
    ax.plot(df[param], df["pass_at_1"], marker="o")
    ax.set_ylim(0, 1)
    ax.set_xlabel(param)
    ax.set_ylabel("Pass@1")
    ax.set_title(f"Pass@1 vs {title}")

plt.tight_layout()
plt.show()